# Поиск бабочек и молей

Сравниваем готовые модели по качеству рамок и времени обработки. Проверяем влияние порога Grounding DINO tiny и дополнительно сравниваем две конфигурации Grounding DINO Base.


In [11]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib.patches import Rectangle

from butterfly_data import ROOT, atomic_write, indexed, jsonl, open_image, read_boxes, run_inference, team_data, write_json
from butterfly_eval import load_saved_predictions, metrics, per_photo_metrics, ranking_key
from butterfly_models import benchmark_configs, resolve_device

USE_SAVED = True
SAVED_RESULTS = ROOT / "model_results"
SMOKE_TEST = False
DEVICE = "cuda"  # Нужен только для инференса.
reuse = USE_SAVED and not SMOKE_TEST
work = ROOT / "data/benchmark"
out = work / "smoke_results" if SMOKE_TEST else ROOT / "model_results"
image_dir = ROOT / "data/images"
photos, gold, reviews = team_data(images_dir=image_dir if image_dir.is_dir() else None)
photos = photos[:3] if SMOKE_TEST else photos
gold = {p.photo_id: gold[p.photo_id] for p in photos}
groups = benchmark_configs()
print(f"{len(photos)} фото · {sum(map(len, gold.values()))} рамок")

400 фото · 361 рамок


## Дополнительный эксперимент

Снижаем порог Grounding DINO, чтобы уменьшить пропуски. Проверяем, компенсирует ли это рост ложных срабатываний. Остальные настройки сохраняем.

In [12]:
def infer_and_save(group):
    rows, runtime = run_inference(photos, group, resolve_device(DEVICE), work / "runs", work / "images")
    for config in group:
        atomic_write(out / f"{config.name}.jsonl", jsonl(rows[config.name]))
        write_json(out / f"{config.name}.json", {"model": config.to_dict(), "runtime": runtime})
    return rows


if reuse:
    predictions, configs = load_saved_predictions(SAVED_RESULTS)
else:
    configs = {cfg.name: cfg for group in groups.values() for cfg in group}
    predictions = {}
    for group in groups.values():
        predictions.update(infer_and_save(group))

## Сравнение

Считаем object F1, precision, recall и ошибки на одной разметке. Для DINO Base сравниваем исходный вариант без NMS с повышенными порогами и NMS. Время Base измерено на RTX 5060 Laptop, остальных моделей на RTX A4000. Выбираем модель по F1.


In [ ]:
details = {name: per_photo_metrics(gold, rows) for name, rows in predictions.items()}
accepted = [pid for pid in gold if reviews[pid]["review_resolution"] == "ACCEPTED"]
slices = {
    "все фото": list(gold),
    "принятые на ревью": accepted,
    "без рамок": [pid for pid, boxes in gold.items() if not boxes],
    "одна рамка": [pid for pid, boxes in gold.items() if len(boxes) == 1],
    "несколько рамок": [pid for pid, boxes in gold.items() if len(boxes) > 1],
    "мелкие объекты": [pid for pid, boxes in gold.items()
                       if any((x2 - x1) * (y2 - y1) <= 0.05 for x1, y1, x2, y2 in boxes)],
}
report = pd.DataFrame([
    {"model": name, "target": configs[name].target, "slice": label, **metrics(rows, ids)}
    for name, rows in details.items() for label, ids in slices.items()
])
if reuse:
    for name in predictions:
        saved = json.loads((SAVED_RESULTS / f"{name}.json").read_text())
        duration = saved.get("runtime", {}).get("total_seconds")
        if duration is not None:
            report.loc[(report["model"] == name) & (report["slice"] == "все фото"), "inference_seconds"] = duration

overall = report[report["slice"] == "все фото"].sort_values(
    ["f1_iou50", "inference_seconds", "model"], ascending=[False, True, True],
)
display(overall.drop(columns="slice").round(4))
display(report[report["slice"] != "все фото"][
    ["model", "slice", "photos", "precision", "recall", "f1_iou50", "f1_iou75", "fp", "fn"]
].round(4))
atomic_write(out / "leaderboard.csv", report.to_csv(index=False).encode())

if SMOKE_TEST:
    if overall["technical_errors"].any():
        raise RuntimeError(f"Проверьте журнал в {work / 'runs'}")
    print("Быстрая проверка завершена. Для полного сравнения задайте SMOKE_TEST = False.")
else:
    best = overall.iloc[0]
    print(f"Лидер среди измеренных конфигураций — {best['model']} (F1@0,50 {best['f1_iou50']:.4f}).")
    eligible = overall[overall["model"].map(lambda name: configs[name].includes_moths)
                       & (overall["technical_errors"] == 0)]
    if eligible.empty:
        print("Для экспорта нужна успешная конфигурация с молями. Можно повторить только лидера.")
    else:
        winner = configs[eligible.iloc[0]["model"]]
        write_json(out / "winner_config.json", winner.to_dict())
        print(f"Выбрана модель — {winner.name}.")

In [ ]:
base, low = (config.name for config in groups["grounding_dino"])
if {base, low} <= predictions.keys():
    columns = ["f1_iou50", "precision", "recall", "tp", "fp", "fn", "fp_on_negative"]
    dino = overall.set_index("model").loc[[base, low], columns]
    display(dino.round(4))
    delta = dino.loc[low] - dino.loc[base]
    print(f"Порог 0,30 → 0,20: ΔF1 {delta['f1_iou50']:+.4f}, ΔTP {delta['tp']:+.0f}, ΔFP {delta['fp']:+.0f}.")

## Примеры ошибок

Бирюзовым показан эталон, оранжевым — предсказания. Показываем по одному кадру с лишней или пропущенной рамкой, когда такие случаи есть.

In [15]:
if not reuse:
    rows_by_id = {name: indexed(rows) for name, rows in predictions.items()}
    order = sorted(gold, key=lambda pid: (pid not in accepted, pid))
    examples = [
        next((pid for pid in order if is_positive == bool(gold[pid])
              and any(rows[pid][key] for rows in details.values())), None)
        for is_positive, key in [(False, "fp"), (True, "fn")]
    ]
    for pid in (pid for pid in examples if pid is not None):
        photo = next(photo for photo in photos if photo.photo_id == pid)
        image = open_image(photo, work / "images")
        width, height = image.size
        fig, axes = plt.subplots(1, len(configs), figsize=(16, 4))
        for ax, name in zip(axes, configs):
            ax.imshow(image)
            row = rows_by_id[name][pid]
            for boxes, color in [(gold[pid], "#23dbb6"), (read_boxes(row["boxes"] or []), "#ff9c38")]:
                for x1, y1, x2, y2 in boxes:
                    ax.add_patch(Rectangle((x1 * width, y1 * height), (x2-x1) * width, (y2-y1) * height,
                                           fill=False, edgecolor=color, lw=2))
            scores = details[name][pid]
            ax.set_title(f"{name}\nFP {scores['fp']} · FN {scores['fn']}", fontsize=10)
            ax.axis("off")
        fig.suptitle(f"{pid} · {reviews[pid]['review_resolution']}")
        plt.show()